[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/cours/seance4_cours.ipynb)

# Séance 3.4 — Régression linéaire — expliquer, et de combien

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'étude de cas en binôme)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- ajuster une régression avec `smf.ols("y ~ x", donnees).fit()`
- lire un coefficient, sa p-value et son intervalle de confiance
- dire ce que le R² mesure — et ce qu'il ne mesure pas
- interpréter un coefficient « toutes choses égales par ailleurs »
- faire entrer une variable qualitative dans un modèle
- reconnaître une extrapolation et refuser d'y répondre

## De « il y a un lien » à « de combien »

En séance 3.3, vous avez établi que le nombre de références et le montant
d'une commande sont liés. Le directeur commercial pose la question suivante :

> *« Si nos vendeurs poussent **une référence de plus** dans chaque panier,
> ça rapporte combien ? »*

Une corrélation ne peut pas répondre : c'est un nombre sans unité. La
**régression** répond en euros.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

print(cmd.shape)
cmd.head(3)

## 1. Une droite qui résume le nuage

La régression cherche la droite qui passe **au plus près** de tous les points.
Elle s'écrit avec un `~`, qui se lit « expliquée par » :

In [ ]:
m = smf.ols("ca ~ nart", cmd).fit()

print(m.summary().tables[1])

Deux lignes, six colonnes. Pour l'instant, regardez-en **deux** :

- **`coef` de `nart` : 15,93.** Chaque référence supplémentaire dans un panier
  correspond à **+15,93 € de commande**. Voilà la réponse au directeur
  commercial.
- **`P>|t|` : 0,000.** La p-value de la séance 3.2, appliquée au coefficient.
  Sous 0,05 : ce coefficient n'est pas un artefact du hasard.

L'`Intercept` (221,95 €) est le montant prédit pour une commande de **zéro**
référence. Aucun sens commercial — c'est normal, l'intercept sert à caler la
droite, pas à être interprété.

### Les quatre autres colonnes

| Colonne | Ce qu'elle dit |
|---|---|
| `std err` | la précision de l'estimation : 0,87 € ici |
| `t` | le coefficient rapporté à son erreur ; sert à calculer la p-value |
| `[0.025` `0.975]` | l'intervalle de confiance : le vrai effet est plausiblement entre **14,22 €** et **17,65 €** |

Un coefficient s'annonce toujours avec sa fourchette : « environ 16 €, entre
14 et 18 ». Jamais « 15,9344 € ».

## 2. Le R² — ce qu'il dit, ce qu'il ne dit pas

In [ ]:
print("R2 :", round(m.rsquared, 3))

**0,146.** Le nombre de références reproduit **15 %** de la variation des
montants. Les 85 % restants viennent d'ailleurs : le prix des articles, les
quantités, le type de client.

> ⚠️ **Un R² faible ne rend pas le coefficient faux.** Ici : 15 % d'explication
> et un effet mesuré à ±1,7 € près. Ce sont deux questions différentes :
>
> - le **coefficient** répond à « de combien ? »
> - le **R²** répond à « est-ce que ma variable suffit à prédire ? »
>
> On peut très bien mesurer précisément un effet réel mais petit.

## 3. Plusieurs variables — et le coefficient qui change

Ajoutons le nombre d'articles. Avant d'exécuter : à votre avis, le coefficient
de `nart` va-t-il monter, descendre, ou rester à 15,93 ?

In [ ]:
m2 = smf.ols("ca ~ nart + qte", cmd).fit()

# Version etroite du tableau : deux colonnes suffisent pour l'essentiel
pd.DataFrame({"coef": m2.params.round(2), "p": m2.pvalues.round(3)})

**15,93 € → 2,05 €.** Divisé par huit, sans qu'on ait touché aux données.

Pourquoi ? Les commandes qui portent sur beaucoup de références contiennent
aussi beaucoup d'articles. Dans le premier modèle, `nart` récupérait **tout
le mérite** : le sien et celui des quantités. Le second modèle sépare les
deux.

Les deux coefficients sont justes. Ils ne répondent simplement pas à la même
question :

| Modèle | Ce que dit le coefficient de `nart` |
|---|---|
| `ca ~ nart` | une commande avec une référence de plus vaut 15,93 € de plus |
| `ca ~ nart + qte` | **à nombre d'articles égal**, une référence de plus vaut 2,05 € |

> ⚠️ **Un coefficient ne se lit jamais seul.** Il se lit « toutes choses égales
> par ailleurs », et le « par ailleurs » est exactement la liste des variables
> du modèle. Deux études sérieuses peuvent publier des chiffres différents pour
> cette raison, sans qu'aucune ne se trompe.

Et le R² ?

In [ ]:
print("R2 simple  :", round(m.rsquared, 3))
print("R2 complet :", round(m2.rsquared, 3))

De 0,146 à 0,721. La deuxième variable apportait vraiment quelque chose.

## 4. Une variable qualitative dans le modèle

`pays` est du texte. La régression le transforme en comparaisons : une
modalité sert de **référence**, les autres se lisent par rapport à elle.

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

m3 = smf.ols("ca ~ qte + pays", sub).fit()
pd.DataFrame({"coef": m3.params.round(2), "p": m3.pvalues.round(3)})

L'Allemagne n'apparaît pas : c'est la **référence** (première par ordre
alphabétique). Tout se lit par rapport à elle, à quantité d'articles égale :

- **Royaume-Uni : −99,59 €, p = 0,001.** À panier d'articles identique, une
  commande britannique vaut cent euros de moins qu'une allemande. Solide.
- **France : −27,32 €, p = 0,448.** Rien ne distingue la France de
  l'Allemagne — c'est très exactement la conclusion de la séance 3.2, obtenue
  cette fois en tenant compte des quantités.
- **Irlande : +34,35 €, p = 0,346.** Le fameux écart irlandais **disparaît**
  une fois les quantités prises en compte : les commandes irlandaises ne sont
  pas plus chères par référence, elles sont simplement **plus grosses**.

C'est le genre de phrase qu'on ne peut écrire qu'avec une régression.

## 5. Prédire — et jusqu'où

In [ ]:
nouvelles = pd.DataFrame({"nart": [20]})

print("commande de 20 references :", m.predict(nouvelles).round(2).iloc[0], "euros")

540,63 € — soit `221,95 + 20 × 15,93`. Une régression est une machine à
prédire autant qu'à expliquer.

Maintenant, la même machine, hors du domaine observé.

In [ ]:
absurde = pd.DataFrame({"nart": [500]})

print("commande de 500 references :", m.predict(absurde).round(2).iloc[0], "euros")
print("maximum reellement observe :", cmd["nart"].max(), "references")

**8 189 €**, annoncés sans la moindre réserve — pour une commande de 500
références alors que la plus fournie du fichier en compte **259**.

Le modèle a prolongé sa droite dans une zone où **il n'a jamais rien vu**.
Rien dans les données ne dit que la relation reste droite là-bas ; elle
pourrait plafonner, ou s'effondrer.

> ⚠️ **L'extrapolation est l'erreur silencieuse par excellence.** Le résultat
> a l'air d'un résultat. Avant toute prédiction, vérifiez que vos valeurs
> d'entrée tombent **dans** l'intervalle observé.

## 6. L'erreur bruyante

In [ ]:
smf.ols("ca ~ nart", cmd).summary()

Dernière ligne :

```
AttributeError: 'OLS' object has no attribute 'summary'
```

Traduction : l'objet créé par `smf.ols(...)` est un **modèle non ajusté**. Il
n'a pas encore de résultats à résumer. Il manque `.fit()` :

```python
smf.ols("ca ~ nart", cmd).fit().summary()
```

Vous ferez cette erreur. Tout le monde la fait. Elle se répare en cinq
caractères.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| ajuster un modèle | `m = smf.ols("ca ~ nart", cmd).fit()` |
| le tableau des coefficients | `print(m.summary().tables[1])` |
| une version étroite | `pd.DataFrame({"coef": m.params.round(2), "p": m.pvalues.round(3)})` |
| le R² | `m.rsquared` |
| plusieurs variables | `smf.ols("ca ~ nart + qte", cmd)` |
| une variable qualitative | `smf.ols("ca ~ qte + pays", cmd)` |
| prédire | `m.predict(pd.DataFrame({"nart": [20]}))` |

## Lire un tableau de régression

| Colonne | Ce qu'elle dit |
|---|---|
| `coef` | de combien `y` bouge quand `x` augmente d'une unité, **les autres variables restant fixes** |
| `std err` | la précision de cette estimation |
| `P>|t|` | la p-value : sous 0,05, on retient le coefficient |
| `[0.025 0.975]` | l'intervalle de confiance du coefficient |

## Les quatre phrases à retenir

1. **Un coefficient se lit toujours « à autres variables constantes ».** Seul,
   `nart` valait 15,93 € ; avec `qte` dans le modèle, 2,05 €. Les deux sont
   justes, ils ne répondent pas à la même question.

2. **Un R² faible n'invalide pas un coefficient.** `ca ~ nart` explique 15 %
   de la variation et son coefficient est parfaitement mesuré.

3. **Une régression ne démontre pas une causalité.** Elle mesure une
   association, en tenant compte des variables qu'on lui a données — et
   d'aucune autre.

4. **Hors du domaine observé, un modèle invente.** Il répondra quand même,
   sans prévenir.